In [17]:
"""import pandas as pd
import numpy as np
from llama_index.core import Document, VectorStoreIndex

# -----------------------------
# Sample dataset (100 rows)
# -----------------------------
np.random.seed(42)

n = 100
categories = ["Electronics", "Clothing", "Home", "Sports", "Beauty"]

df = pd.DataFrame({
    "product_id": range(1, n + 1),
    "product_name": [f"Product_{i}" for i in range(1, n + 1)],
    "category": np.random.choice(categories, n),
    "price": np.round(np.random.uniform(10, 500, n), 2),
    "stock": np.random.randint(0, 200, n),
    "units_sold": np.random.randint(0, 1000, n)
})

# -----------------------------
# IMPORTANT OPTIMIZATION
# Only index relevant columns + small representation
# -----------------------------
text_data = df[["product_name", "price"]].to_string(index=False)

doc = Document(text=text_data)

index = VectorStoreIndex.from_documents([doc])

query_engine = index.as_query_engine(similarity_top_k=2)

response = query_engine.query(
    "Which 5 products have the highest price?"
)

print(response)"""

'import pandas as pd\nimport numpy as np\nfrom llama_index.core import Document, VectorStoreIndex\n\n# -----------------------------\n# Sample dataset (100 rows)\n# -----------------------------\nnp.random.seed(42)\n\nn = 100\ncategories = ["Electronics", "Clothing", "Home", "Sports", "Beauty"]\n\ndf = pd.DataFrame({\n    "product_id": range(1, n + 1),\n    "product_name": [f"Product_{i}" for i in range(1, n + 1)],\n    "category": np.random.choice(categories, n),\n    "price": np.round(np.random.uniform(10, 500, n), 2),\n    "stock": np.random.randint(0, 200, n),\n    "units_sold": np.random.randint(0, 1000, n)\n})\n\n# -----------------------------\n# IMPORTANT OPTIMIZATION\n# Only index relevant columns + small representation\n# -----------------------------\ntext_data = df[["product_name", "price"]].to_string(index=False)\n\ndoc = Document(text=text_data)\n\nindex = VectorStoreIndex.from_documents([doc])\n\nquery_engine = index.as_query_engine(similarity_top_k=2)\n\nresponse = quer

In [16]:
"""User question (natural language)
        ↓
LLM decides:
   ├── SQL path (structured/numeric questions)
   └── RAG path (semantic/text questions)
        ↓
DuckDB OR LlamaIndex retrieval
        ↓
LLM final answer synthesis"""

'User question (natural language)\n        ↓\nLLM decides:\n   ├── SQL path (structured/numeric questions)\n   └── RAG path (semantic/text questions)\n        ↓\nDuckDB OR LlamaIndex retrieval\n        ↓\nLLM final answer synthesis'

In [19]:
import pandas as pd
import numpy as np
import duckdb
import re

from llama_index.core import Document, VectorStoreIndex
from openai import OpenAI

# -----------------------------
# 1. OpenAI client
# -----------------------------
client = OpenAI()

# -----------------------------
# 2. Dataset
# -----------------------------
np.random.seed(42)

n = 100

df = pd.DataFrame({
    "product_id": range(1, n + 1),
    "product_name": [f"Product_{i}" for i in range(1, n + 1)],
    "category": np.random.choice(["Electronics", "Clothing", "Home"], n),
    "price": np.random.uniform(10, 500, n),
    "description": [
        "High quality modern product" if i % 2 == 0
        else "Budget friendly everyday item"
        for i in range(n)
    ]
})

# -----------------------------
# 3. DuckDB setup (SQL engine)
# -----------------------------
con = duckdb.connect("my_db.duckdb")
con.register("products", df)

# -----------------------------
# 4. LlamaIndex (semantic layer)
# -----------------------------
docs = [
    Document(text=f"{row['product_name']} - {row['description']}")
    for _, row in df.iterrows()
]

index = VectorStoreIndex.from_documents(docs)
rag_engine = index.as_query_engine()

# -----------------------------
# 5. CLEAN SQL FUNCTION (FIXES YOUR ERROR)
# -----------------------------
def clean_sql(sql: str) -> str:
    sql = re.sub(r"```sql", "", sql)
    sql = re.sub(r"```", "", sql)
    return sql.strip()

# -----------------------------
# 6. ROUTER (SQL vs RAG)
# -----------------------------
def route_query(question: str) -> str:
    prompt = f"""
You are a router.

Decide:
- SQL → numbers, ranking, filtering, aggregation
- RAG → meaning, description, similarity

Return ONLY one word: SQL or RAG

Question:
{question}
"""

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content.strip()

# -----------------------------
# 7. SQL EXECUTION PATH
# -----------------------------
def run_sql(question: str):
    prompt = f"""
Convert to DuckDB SQL.

Table: products(product_name, price, category)

Rules:
- ONLY SQL
- NO markdown
- NO explanations

Question:
{question}
"""

    sql = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    ).choices[0].message.content

    # 🔥 FIX: remove ```sql blocks
    sql = clean_sql(sql)

    result = con.execute(sql).df()
    return result

# -----------------------------
# 8. RAG EXECUTION PATH
# -----------------------------
def run_rag(question: str):
    return rag_engine.query(question)

# -----------------------------
# 9. FINAL ANSWER ENGINE
# -----------------------------
def answer(question: str):

    mode = route_query(question)

    if mode == "SQL":
        result = run_sql(question)

        prompt = f"""
Question: {question}

Data:
{result.to_string(index=False)}

Explain clearly and concisely.
"""
    else:
        result = run_rag(question)

        prompt = f"""
Question: {question}

Context:
{result}

Explain clearly.
"""

    final = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return final.choices[0].message.content

# -----------------------------
# 10. TESTS
# -----------------------------
print(answer("name top 5 cheapest products"))
print(answer("what do budget products usually look like"))

Based on the provided data, the top 5 cheapest products along with their prices are:

1. **Product_2** - $12.71
2. **Product_58** - $13.41
3. **Product_28** - $22.46
4. **Product_30** - $25.40
5. **Product_75** - $28.07

These products are listed in order of their price, starting from the cheapest.
Budget products typically have several defining characteristics:

1. **Basic Features**: They focus on delivering the essential functions that the product is intended for, without any extra or advanced features. For instance, a budget smartphone might have basic calling and messaging functions but may lack high-end cameras or large storage capacity.

2. **Simpler Design**: The design of budget products tends to be straightforward and utilitarian. They often have a no-frills appearance, with less emphasis on aesthetics or trendy designs. This is in contrast to premium products that may feature sleek designs or luxurious materials.

3. **Lower Quality Materials**: While budget products are mad